# 3. Построение и сравнение моделей

Входные данные: подготовленные сплиты из `data/train_test/`.

Структура раздела:
- 3.1 Загрузка данных
- 3.2 Baseline — Logistic Regression
- 3.3 Основная модель — CatBoost
- 3.4 Подбор гиперпараметров CatBoost (Optuna)
- 3.5 Сравнение моделей

**Метрики оценки:** ROC-AUC, F1, Precision, Recall.
Accuracy не используем ввиду дисбаланса классов (1:3.9).

## 3.1 Загрузка данных

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import (roc_auc_score, f1_score, precision_score,
                             recall_score, classification_report,
                             confusion_matrix, roc_curve)
import joblib
import warnings
warnings.filterwarnings('ignore')

In [2]:
# Scaled — для Logistic Regression
X_train = pd.read_csv('../data/train_test/X_train.csv')
X_test  = pd.read_csv('../data/train_test/X_test.csv')

# Raw — для CatBoost
X_train_raw = pd.read_csv('../data/train_test/X_train_raw.csv')
X_test_raw  = pd.read_csv('../data/train_test/X_test_raw.csv')

y_train = pd.read_csv('../data/train_test/y_train.csv').squeeze()
y_test  = pd.read_csv('../data/train_test/y_test.csv').squeeze()

print(f"Train: {X_train.shape}, Test: {X_test.shape}")
print(f"Доля оттока в train: {y_train.mean():.3f}")
print(f"Доля оттока в test:  {y_test.mean():.3f}")

Train: (12000, 12), Test: (3000, 12)
Доля оттока в train: 0.204
Доля оттока в test:  0.204


## 3.2 Вспомогательная функция оценки моделей

Чтобы не дублировать код метрик для каждой модели,
вынесем оценку в единую функцию.

In [3]:
def evaluate_model(model, X_train, y_train, X_test, y_test, model_name='Model', threshold=0.5):
    from sklearn.metrics import average_precision_score

    y_prob_train = model.predict_proba(X_train)[:, 1]
    y_prob_test  = model.predict_proba(X_test)[:, 1]
    y_pred_train = (y_prob_train >= threshold).astype(int)
    y_pred_test  = (y_prob_test >= threshold).astype(int)

    def _calc(y_true, y_prob, y_pred):
        return {
            'ROC-AUC': round(roc_auc_score(y_true, y_prob), 4),
            'AUC-PR':  round(average_precision_score(y_true, y_prob), 4),
            'F1':      round(f1_score(y_true, y_pred), 4),
            'Prec':    round(precision_score(y_true, y_pred), 4),
            'Recall':  round(recall_score(y_true, y_pred), 4),
        }

    train_m = _calc(y_train, y_prob_train, y_pred_train)
    test_m  = _calc(y_test,  y_prob_test,  y_pred_test)

    print(f"\n{'='*50}")
    print(f"  {model_name}")
    print(f"{'='*50}")
    print(f"  {'':12} {'Train':>10} {'Test':>10} {'Gap':>10}")
    print(f"  {'-'*42}")
    for k in train_m:
        gap = round(train_m[k] - test_m[k], 4)
        flag = ' !!' if abs(gap) > 0.05 else ''
        print(f"  {k:<12} {train_m[k]:>10} {test_m[k]:>10} {gap:>+10}{flag}")

    print(f"\n  Test classification report:")
    print(f"{classification_report(y_test, y_pred_test, target_names=['Остался', 'Ушёл'])}")

    return {'train': train_m, 'test': test_m}, y_prob_test

## 3.3 Baseline — Logistic Regression

Логистическая регрессия используется как baseline-модель.
Она проста, интерпретируема и задаёт нижнюю планку качества —
любая более сложная модель должна её превосходить.
`class_weight='balanced'` компенсирует дисбаланс классов 1:3.9.

In [4]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_val_score

lr = LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42)
lr.fit(X_train, y_train)

# Кросс-валидация
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_scores = cross_val_score(lr, X_train, y_train, cv=cv, scoring='roc_auc')
print(f"CV ROC-AUC: {cv_scores.mean():.4f} ± {cv_scores.std():.4f}")

lr_metrics, lr_probs = evaluate_model(lr, X_train, y_train, X_test, y_test, 'Logistic Regression')

CV ROC-AUC: 0.8884 ± 0.0083

  Logistic Regression
                    Train       Test        Gap
  ------------------------------------------
  ROC-AUC          0.8891     0.8823    +0.0068
  AUC-PR           0.7126     0.6987    +0.0139
  F1               0.6537      0.641    +0.0127
  Prec             0.5499      0.533    +0.0169
  Recall           0.8058     0.8039    +0.0019

  Test classification report:
              precision    recall  f1-score   support

     Остался       0.94      0.82      0.88      2388
        Ушёл       0.53      0.80      0.64       612

    accuracy                           0.82      3000
   macro avg       0.74      0.81      0.76      3000
weighted avg       0.86      0.82      0.83      3000



## 3.4 Основная модель — CatBoost

CatBoost выбран как основная модель по следующим причинам:
- устойчив к выбросам и не требует масштабирования
- эффективно работает на табличных данных
- встроенная обработка дисбаланса классов через `auto_class_weights`
- не требует OHE для категориальных признаков (но у нас уже закодировано)

In [5]:
from catboost import CatBoostClassifier

cb = CatBoostClassifier(
    iterations=500,
    learning_rate=0.05,
    depth=6,
    auto_class_weights='Balanced',
    random_seed=42,
    verbose=100
)
cb.fit(X_train_raw, y_train, eval_set=(X_test_raw, y_test))

# Кросс-валидация
cv_scores_cb = cross_val_score(cb, X_train_raw, y_train, cv=cv, scoring='roc_auc')
print(f"\nCV ROC-AUC: {cv_scores_cb.mean():.4f} ± {cv_scores_cb.std():.4f}")

cb_metrics, cb_probs = evaluate_model(cb, X_train_raw, y_train, X_test_raw, y_test, 'CatBoost')

0:	learn: 0.6535274	test: 0.6538233	best: 0.6538233 (0)	total: 58.1ms	remaining: 29s
100:	learn: 0.3021310	test: 0.3293430	best: 0.3293396 (97)	total: 254ms	remaining: 1s
200:	learn: 0.2796931	test: 0.3277394	best: 0.3268806 (162)	total: 441ms	remaining: 656ms
300:	learn: 0.2603219	test: 0.3309045	best: 0.3268806 (162)	total: 631ms	remaining: 417ms
400:	learn: 0.2438638	test: 0.3331088	best: 0.3268806 (162)	total: 819ms	remaining: 202ms
499:	learn: 0.2303624	test: 0.3373610	best: 0.3268806 (162)	total: 1s	remaining: 0us

bestTest = 0.3268806475
bestIteration = 162

Shrink model to first 163 iterations.
0:	learn: 0.6549898	total: 1.35ms	remaining: 673ms
100:	learn: 0.2993599	total: 138ms	remaining: 543ms
200:	learn: 0.2732640	total: 272ms	remaining: 405ms
300:	learn: 0.2525813	total: 426ms	remaining: 282ms
400:	learn: 0.2340270	total: 572ms	remaining: 141ms
499:	learn: 0.2178632	total: 710ms	remaining: 0us
0:	learn: 0.6570956	total: 1.37ms	remaining: 684ms
100:	learn: 0.3010876	total: 1

## 3.6 Подбор гиперпараметров — Optuna

Optuna использует байесовскую оптимизацию — в отличие от GridSearch
она не перебирает все комбинации подряд, а направленно ищет лучшие
параметры на основе предыдущих попыток. Это эффективнее при большом
пространстве поиска.

Оптимизируем по ROC-AUC на стратифицированной кросс-валидации (3 фолда)
чтобы не переобучиться на тестовой выборке.

In [6]:
import optuna
from catboost import CatBoostClassifier
from sklearn.model_selection import StratifiedKFold, cross_val_score

optuna.logging.set_verbosity(optuna.logging.WARNING)

def objective(trial):
    params = {
        'iterations':        trial.suggest_int('iterations', 200, 1000),
        'learning_rate':     trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        'depth':             trial.suggest_int('depth', 4, 10),
        'l2_leaf_reg':       trial.suggest_float('l2_leaf_reg', 1, 10),
        'bagging_temperature': trial.suggest_float('bagging_temperature', 0, 1),
        'random_strength':   trial.suggest_float('random_strength', 0, 1),
        'auto_class_weights': 'Balanced',
        'random_seed':       42,
        'verbose':           False
    }

    model = CatBoostClassifier(**params)
    cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
    scores = cross_val_score(model, X_train_raw, y_train, cv=cv, scoring='roc_auc')
    return scores.mean()

study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=50, show_progress_bar=True)

print(f"\nЛучший ROC-AUC: {study.best_value:.4f}")
print(f"Лучшие параметры:\n{study.best_params}")

Best trial: 28. Best value: 0.936736: 100%|██████████| 50/50 [01:38<00:00,  1.98s/it]


Лучший ROC-AUC: 0.9367
Лучшие параметры:
{'iterations': 360, 'learning_rate': 0.019897347947253705, 'depth': 4, 'l2_leaf_reg': 9.929714159478038, 'bagging_temperature': 0.3784348838032836, 'random_strength': 0.16730624920542103}


## 3.7 Финальная модель CatBoost

Обучаем модель на лучших параметрах найденных Optuna.
Прирост ROC-AUC после тюнинга: 0.9319 → 0.9367 (+0.48%).
Небольшой прирост говорит о том, что дефолтные параметры CatBoost
уже были близки к оптимальным для данного датасета.

In [7]:
best_params = study.best_params
best_params['auto_class_weights'] = 'Balanced'
best_params['random_seed'] = 42
best_params['verbose'] = 100

cb_tuned = CatBoostClassifier(**best_params)
cb_tuned.fit(X_train_raw, y_train, eval_set=(X_test_raw, y_test))

# Финальная оценка
cb_tuned_metrics, cb_tuned_probs = evaluate_model(
    cb_tuned, X_train_raw, y_train, X_test_raw, y_test, 'CatBoost (tuned)'
)

# Кросс-валидация
cv_final = cross_val_score(cb_tuned, X_train_raw, y_train, cv=cv, scoring='roc_auc')
print(f"\nCV ROC-AUC финальной модели: {cv_final.mean():.4f} ± {cv_final.std():.4f}")

# И сохраняем
joblib.dump(cb_tuned, '../models/catboost_model.pkl')
print("\nМодель сохранена в models/catboost_model.pkl")

0:	learn: 0.6775159	test: 0.6775293	best: 0.6775293 (0)	total: 1.87ms	remaining: 673ms
100:	learn: 0.3452575	test: 0.3527691	best: 0.3527691 (100)	total: 141ms	remaining: 360ms
200:	learn: 0.3208880	test: 0.3337760	best: 0.3337760 (200)	total: 279ms	remaining: 221ms
300:	learn: 0.3117003	test: 0.3293262	best: 0.3293262 (300)	total: 414ms	remaining: 81.2ms
359:	learn: 0.3079691	test: 0.3285593	best: 0.3285271 (352)	total: 496ms	remaining: 0us

bestTest = 0.3285270849
bestIteration = 352

Shrink model to first 353 iterations.

  CatBoost (tuned)
                    Train       Test        Gap
  ------------------------------------------
  ROC-AUC          0.9429     0.9346    +0.0083
  AUC-PR           0.8524     0.8187    +0.0337
  F1               0.7329     0.7137    +0.0192
  Prec             0.6386     0.6144    +0.0242
  Recall           0.8598     0.8513    +0.0085

  Test classification report:
              precision    recall  f1-score   support

     Остался       0.96      0.

## 3.8 Сравнение нескольких моделей

Обучаем и оцениваем несколько моделей за раз:
- Logistic Regression (baseline, scaled)
- Decision Tree
- Random Forest
- CatBoost (tuned)

Для каждой модели выводим метрики train/test и gap для выявления переобучения.

In [8]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
import pandas as pd

models = {
    'Logistic Regression': (LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42), X_train, X_test),
    'Decision Tree': (DecisionTreeClassifier(class_weight='balanced', max_depth=8, min_samples_leaf=20, random_state=42), X_train, X_test),
    'Random Forest': (RandomForestClassifier(n_estimators=300, max_depth=8, class_weight='balanced', random_state=42, n_jobs=-1), X_train, X_test),
    'CatBoost (tuned)': (cb_tuned, X_train_raw, X_test_raw),
}

all_results = {}
for name, (model, Xtr, Xte) in models.items():
    model.fit(Xtr, y_train)
    result, _ = evaluate_model(model, Xtr, y_train, Xte, y_test, name)
    all_results[name] = result

# Сводная таблица
rows = []
for name, res in all_results.items():
    for split in ['train', 'test']:
        row = {'Model': name, 'Split': split}
        row.update(res[split])
        rows.append(row)
    gap_row = {'Model': name, 'Split': 'GAP'}
    for m in res['train']:
        gap_row[m] = round(res['train'][m] - res['test'][m], 4)
    rows.append(gap_row)

summary_df = pd.DataFrame(rows)
print('\n' + '='*70)
print('  ИТОГОВАЯ СВОДНАЯ ТАБЛИЦА')
print('='*70)
print(summary_df.to_string(index=False))


  Logistic Regression
                    Train       Test        Gap
  ------------------------------------------
  ROC-AUC          0.8891     0.8823    +0.0068
  AUC-PR           0.7126     0.6987    +0.0139
  F1               0.6537      0.641    +0.0127
  Prec             0.5499      0.533    +0.0169
  Recall           0.8058     0.8039    +0.0019

  Test classification report:
              precision    recall  f1-score   support

     Остался       0.94      0.82      0.88      2388
        Ушёл       0.53      0.80      0.64       612

    accuracy                           0.82      3000
   macro avg       0.74      0.81      0.76      3000
weighted avg       0.86      0.82      0.83      3000


  Decision Tree
                    Train       Test        Gap
  ------------------------------------------
  ROC-AUC          0.9482     0.9186    +0.0296
  AUC-PR           0.8604     0.7788    +0.0816 !!
  F1               0.7375     0.6924    +0.0451
  Prec             0.6412    

## 3.9 Итоговое сравнение всех моделей

| Метрика    | Logistic Regression | CatBoost (default) | CatBoost (tuned) |
|------------|--------------------:|-------------------:|-----------------:|
| ROC-AUC    | 0.8823              | 0.9348             | 0.9347           |
| F1         | 0.6410              | 0.7189             | 0.7169           |
| Precision  | 0.5330              | 0.6248             | 0.6175           |
| Recall     | 0.8039              | 0.8464             | 0.8546           |
| CV ROC-AUC | 0.8884 ± 0.0083     | 0.9319 ± 0.0041    | 0.9368 ± 0.0043  |

**Выводы:**
- Тюнинг дал прирост на CV (0.9319 → 0.9368), но на тестовой выборке
  результаты практически идентичны — модель и без тюнинга была близка к оптимуму
- CatBoost (tuned) показывает лучший Recall (0.855) — важно с точки зрения бизнеса:
  модель реже пропускает реально уходящих клиентов
- Финальная модель — **CatBoost (tuned)**, сохранена в `models/catboost_model.pkl`

**Следующий шаг:** детальная валидация, визуализация и интерпретация модели
в `04_validation.ipynb`.